# Patagonia: Attribution Models vs. Experiment

**Author:** duydnguyenn  
**Date:** April 2026

---

This notebook analyzes Patagonia's digital marketing experiment to evaluate whether naive attribution models provide reliable guidance for marketing budget allocation, or whether experimentally-derived incremental lift is needed for sound decision-making.

Patagonia ran a 3-week holdout study across five channels — Email, YouTube, Instagram, Google Search, and Contextual Display — with 2 million randomly selected customers. Twenty percent of customers per channel were held out from receiving marketing, enabling causal estimation of each channel's true impact.

## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import ttest_ind

pd.set_option('display.float_format', '{:,.4f}'.format)

df = pd.read_csv('data/sample.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (50000, 22)


,age,gender,days_since_last_visit,days_since_last_purchase,num_past_purchases,sales_past_12m,time_on_site_last_visit,time_on_site_12m,email_T,email_touchpoints,...,instagram_T,instagram_touchpoints,google_search_T,google_search_touchpoints,contextual_display_T,contextual_display_touchpoints,sales,touchpoint_sequence,first_touch,last_touch
0,33,Male,8.2200,18.9300,2,166.7600,1.2900,23.1800,1,1,...,1,0,1,0,0,0,0.0000,email,email,email
1,26,Male,16.5900,16.9900,3,22.2700,12.1500,48.5300,1,1,...,0,0,0,0,0,0,0.0000,email,email,email
2,39,Male,7.6700,9.4500,2,6.7000,6.6700,36.8000,1,0,...,1,1,1,0,1,0,0.0000,instagram,instagram,instagram
3,31,Female,9.2800,43.3500,2,98.6700,1.0000,54.2900,1,0,...,1,0,1,0,1,1,0.0000,contextual_display,contextual_display,contextual_display
4,29,Female,0.3800,7.9000,3,132.6900,0.3800,4.5500,1,0,...,1,0,1,0,0,0,0.0000,NaN,NaN,NaN


## Question 1: Attribution Without Incrementality

Before leveraging the experimental design, we apply three standard marketing attribution models to understand what they suggest about each channel's contribution to sales. These models allocate observed sales across the touchpoints in a customer's journey without accounting for what would have happened in the absence of marketing.

### 1a. Last-Touch Attribution

Last-touch attribution assigns 100% of the credit for a conversion to the final channel a customer interacted with before purchasing. This model is simple and widely used, but it systematically favors channels that appear late in the funnel — such as search — regardless of whether those channels were causally responsible for the purchase decision.

In [2]:
converters = df[df['sales'] > 0].copy()

last_touch = converters.groupby('last_touch')['sales'].sum().sort_values(ascending=False).reset_index()
last_touch.columns = ['Channel', 'Attributed Sales ($)']
last_touch['Share (%)'] = (last_touch['Attributed Sales ($)'] / last_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('Last-Touch Attribution')
last_touch

Last-Touch Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"12,383.8500",35.6700
1,google_search,"8,696.3800",25.0500
2,instagram,"8,141.9800",23.4500
3,youtube,"3,576.6700",10.3000
4,email,"1,916.9100",5.5200


### 1b. First-Touch Attribution

First-touch attribution assigns 100% of the credit to the first channel a customer encountered. This model favors awareness-driving channels at the top of the funnel and is often used to justify prospecting spend. Like last-touch, it ignores the full conversion journey and attributes all value to a single touchpoint.

In [3]:
first_touch = converters.groupby('first_touch')['sales'].sum().sort_values(ascending=False).reset_index()
first_touch.columns = ['Channel', 'Attributed Sales ($)']
first_touch['Share (%)'] = (first_touch['Attributed Sales ($)'] / first_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('First-Touch Attribution')
first_touch

First-Touch Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"14,738.4600",42.4500
1,email,"11,417.4600",32.8900
2,youtube,"4,804.8200",13.8400
3,instagram,"2,353.1300",6.7800
4,google_search,"1,401.9200",4.0400


### 1c. Linear Attribution

Linear attribution distributes credit equally across all touchpoints in the customer journey. A customer who interacted with three channels before converting would allocate one-third of their sales value to each. This approach is more balanced than single-touch models but still does not distinguish between channels that drove the conversion and those that were merely present.

In [4]:
channels = ['email', 'youtube', 'instagram', 'google_search', 'contextual_display']

linear_credits = {ch: 0.0 for ch in channels}

for _, row in converters.iterrows():
    seq = str(row['touchpoint_sequence'])
    touched = [ch for ch in channels if ch in seq]
    if len(touched) == 0:
        continue
    credit_per_channel = row['sales'] / len(touched)
    for ch in touched:
        linear_credits[ch] += credit_per_channel

linear_touch = pd.DataFrame(list(linear_credits.items()), columns=['Channel', 'Attributed Sales ($)'])
linear_touch = linear_touch.sort_values('Attributed Sales ($)', ascending=False).reset_index(drop=True)
linear_touch['Share (%)'] = (linear_touch['Attributed Sales ($)'] / linear_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('Linear Attribution')
linear_touch

Linear Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"14,324.2883",41.2600
1,email,"5,934.9367",17.1000
2,instagram,"5,061.6700",14.5800
3,youtube,"4,785.1133",13.7800
4,google_search,"4,609.7817",13.2800


### 1d. Comparing the Three Models

The three models often produce meaningfully different rankings and credit allocations across channels. These differences reflect the models' structural assumptions rather than any underlying truth about marketing effectiveness. For instance, a channel that consistently appears early in the funnel will look strong under first-touch attribution but weak under last-touch, regardless of whether it actually drove conversions.

The critical limitation of all three models is that they attribute 100% of observed sales to marketing — implicitly assuming that no purchases would have occurred organically. This leads to a fundamental overestimate of marketing-driven revenue and can produce misleading budget recommendations.

In [5]:
comparison = last_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'Last-Touch (%)'})
comparison = comparison.merge(first_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'First-Touch (%)'}), on='Channel')
comparison = comparison.merge(linear_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'Linear (%)'}), on='Channel')
print('Attribution Model Comparison — Share of Sales (%)')
comparison.sort_values('Last-Touch (%)', ascending=False)

Attribution Model Comparison — Share of Sales (%)


,Channel,Last-Touch (%),First-Touch (%),Linear (%)
0,contextual_display,35.6700,42.4500,41.2600
1,google_search,25.0500,4.0400,13.2800
2,instagram,23.4500,6.7800,14.5800
3,youtube,10.3000,13.8400,13.7800
4,email,5.5200,32.8900,17.1000


## Question 2: Estimating Incremental Sales with Experimental Data

The holdout experiment allows us to move beyond attribution and measure the true causal effect of each channel. By randomly withholding marketing from 20% of customers per channel, Patagonia created a valid counterfactual: what would these customers have spent without the marketing exposure?

### 2a. Intent-to-Treat (ITT) Effects

The Intent-to-Treat effect measures the average sales difference between the treatment group (customers eligible to receive the channel) and the holdout group (customers withheld from the channel). We use two-sample t-tests to assess statistical significance.

In [6]:
channel_T = {
    'Email': 'email_T',
    'YouTube': 'youtube_T',
    'Instagram': 'instagram_T',
    'Google Search': 'google_search_T',
    'Contextual Display': 'contextual_display_T'
}

itt_results = []
for channel, t_col in channel_T.items():
    treatment = df[df[t_col] == 1]['sales']
    holdout = df[df[t_col] == 0]['sales']
    itt = treatment.mean() - holdout.mean()
    t_stat, p_val = ttest_ind(treatment, holdout)
    itt_results.append({
        'Channel': channel,
        'Treatment Mean ($)': treatment.mean(),
        'Holdout Mean ($)': holdout.mean(),
        'ITT Effect ($)': itt,
        'p-value': p_val,
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

itt_df = pd.DataFrame(itt_results)
print('Intent-to-Treat Effects by Channel')
itt_df

Intent-to-Treat Effects by Channel


,Channel,Treatment Mean ($),Holdout Mean ($),ITT Effect ($),p-value,Significant
0,Email,0.8557,0.6396,0.2161,0.0132,Yes
1,YouTube,0.8285,0.7451,0.0833,0.3418,No
2,Instagram,0.8215,0.7730,0.0485,0.5801,No
3,Google Search,0.8137,0.8043,0.0094,0.9146,No
4,Contextual Display,0.8085,0.8250,-0.0165,0.8497,No


### 2b. Total Incremental Sales

To translate per-customer ITT effects into total incremental revenue, we multiply each channel's ITT by the size of its treatment group. This gives us the total incremental sales attributable to each channel across the experiment population.

In [7]:
incremental_sales = []
for row in itt_results:
    t_col = channel_T[row['Channel']]
    n_treatment = (df[t_col] == 1).sum()
    total_incremental = row['ITT Effect ($)'] * n_treatment
    incremental_sales.append({
        'Channel': row['Channel'],
        'ITT Effect ($)': row['ITT Effect ($)'],
        'Treatment Group Size': n_treatment,
        'Total Incremental Sales ($)': total_incremental
    })

incremental_df = pd.DataFrame(incremental_sales)
print(f"Total Incremental Sales: ${incremental_df['Total Incremental Sales ($)'].sum():,.2f}")
incremental_df

Total Incremental Sales: $13,602.26


,Channel,ITT Effect ($),Treatment Group Size,Total Incremental Sales ($)
0,Email,0.2161,39846,"8,610.7706"
1,YouTube,0.0833,40004,"3,334.0273"
2,Instagram,0.0485,39996,"1,939.3405"
3,Google Search,0.0094,40076,378.0137
4,Contextual Display,-0.0165,39893,-659.8889


### 2c. Comparing Naive Attribution to Incremental Sales

Naive attribution models attribute all observed sales among converting customers to the marketing channels in their journey. The experimental approach, by contrast, isolates only the sales that would not have occurred without the channel. The gap between these two estimates represents organic sales — conversions that would have happened regardless of marketing.

In [8]:
naive_total = last_touch['Attributed Sales ($)'].sum()
incremental_total = incremental_df['Total Incremental Sales ($)'].sum()
organic_estimate = naive_total - incremental_total

print(f'Naive Attribution Total (Last-Touch): ${naive_total:,.2f}')
print(f'Incremental Sales Total:              ${incremental_total:,.2f}')
print(f'Estimated Organic Sales:              ${organic_estimate:,.2f}')
print(f'Naive Overstatement Multiple:         {naive_total / incremental_total:.1f}x')

Naive Attribution Total (Last-Touch): $34,715.79
Incremental Sales Total:              $13,602.26
Estimated Organic Sales:              $21,113.53
Naive Overstatement Multiple:         2.6x


### 2d. Organic Sales and Implications

The difference between naive attribution totals and experimentally measured incremental sales represents conversions that would have occurred in the absence of any of the five channels. These are customers who purchased organically — through direct site visits, word of mouth, or prior brand affinity.

Naive attribution models assign this organic revenue to marketing, creating the illusion of high effectiveness. This overstatement can lead companies to over-invest in channels that appear productive under attribution but generate little to no incremental value. The experimental holdout design is essential for disentangling true lift from organic behavior.

## Question 3: Estimating Incremental Impact Using Regression

Regression provides a flexible framework for estimating the incremental impact of each channel while controlling for pre-existing customer differences. Although random assignment should balance groups by design, regression controls can improve precision and reveal whether heterogeneous customer characteristics moderate channel effectiveness.

### 3a. Simple OLS Regression (Treatment Indicators Only)

We begin with a baseline regression that uses only the five treatment indicators as independent variables. The coefficients on each treatment indicator represent the average incremental sales per customer assigned to that channel — equivalent to the ITT estimates computed above.

In [9]:
model_simple = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T',
    data=df
).fit()

print(model_simple.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.483
Date:                Mon, 06 Apr 2026   Prob (F-statistic):              0.192
Time:                        23:16:37   Log-Likelihood:            -1.7391e+05
No. Observations:               50000   AIC:                         3.478e+05
Df Residuals:                   49994   BIC:                         3.479e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.5405 

### 3b. Regression with Control Variables

Adding customer-level controls — demographics and prior engagement — can reduce residual variance and improve the precision of treatment effect estimates. We include age, gender, recency of visit and purchase, purchase history, and site engagement metrics as controls.

In [10]:
model_controls = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T '
    '+ age + C(gender) + days_since_last_visit + days_since_last_purchase '
    '+ num_past_purchases + sales_past_12m + time_on_site_last_visit + time_on_site_12m',
    data=df
).fit()

print(model_controls.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     5.161
Date:                Mon, 06 Apr 2026   Prob (F-statistic):           2.78e-09
Time:                        23:16:37   Log-Likelihood:            -1.7388e+05
No. Observations:               50000   AIC:                         3.478e+05
Df Residuals:                   49986   BIC:                         3.479e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### 3c. Regression with Interaction Term

We explore whether the effectiveness of Google Search varies with a customer's purchase history. Customers with more past purchases may be more likely to respond to search ads, since they are already familiar with the brand and may be searching with higher purchase intent.

In [11]:
model_interaction = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T '
    '+ age + C(gender) + days_since_last_visit + days_since_last_purchase '
    '+ num_past_purchases + sales_past_12m + time_on_site_last_visit + time_on_site_12m '
    '+ google_search_T:num_past_purchases',
    data=df
).fit()

print(model_interaction.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     5.022
Date:                Mon, 06 Apr 2026   Prob (F-statistic):           1.73e-09
Time:                        23:16:37   Log-Likelihood:            -1.7388e+05
No. Observations:               50000   AIC:                         3.478e+05
Df Residuals:                   49985   BIC:                         3.479e+05
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

### 3d. Comparing Regression Results

We extract the treatment coefficients from all three models to assess whether adding controls or interactions materially changes the estimated channel effects. Under a well-executed random assignment, treatment coefficients should be stable across specifications. Large changes would indicate imbalance in the experimental design or omitted variable bias.

In [12]:
treatment_vars = ['email_T', 'youtube_T', 'instagram_T', 'google_search_T', 'contextual_display_T']

coef_comparison = pd.DataFrame({
    'Variable': treatment_vars,
    'Simple OLS': [model_simple.params.get(v, np.nan) for v in treatment_vars],
    'With Controls': [model_controls.params.get(v, np.nan) for v in treatment_vars],
    'With Interaction': [model_interaction.params.get(v, np.nan) for v in treatment_vars]
})

print('Treatment Coefficient Comparison Across Models')
coef_comparison

Treatment Coefficient Comparison Across Models


,Variable,Simple OLS,With Controls,With Interaction
0,email_T,0.2162,0.2136,0.2119
1,youtube_T,0.0840,0.0828,0.0827
2,instagram_T,0.0484,0.0482,0.0480
3,google_search_T,0.0082,0.0071,-0.2170
4,contextual_display_T,-0.0168,-0.0162,-0.0166


## Question 4: Calculating ROI of Marketing Channels

Return on investment (ROI) requires pairing each channel's incremental revenue with its actual cost. We calculate total spend per channel using the provided cost-per-touchpoint rates and the observed number of touchpoints in the sample, then scale to the full experiment population.

### 4a. Channel Cost Calculations

Cost rates per channel:
- **Email:** $0.12 per email sent
- **YouTube:** $0.20 per completed 30-second view
- **Instagram:** $0.10 per in-stream view (3+ seconds)
- **Google Search:** $2.30 per click
- **Contextual Display:** $0.02 per viewable impression (per 20 vCPM = $0.001 per impression)

In [13]:
cost_per_touchpoint = {
    'email': 0.12,
    'youtube': 0.20,
    'instagram': 0.10,
    'google_search': 2.30,
    'contextual_display': 0.02 / 20
}

scale_factor = 2_000_000 / len(df)

cost_rows = []
for ch, rate in cost_per_touchpoint.items():
    touchpoints_col = f'{ch}_touchpoints'
    total_touchpoints_sample = df[touchpoints_col].sum()
    total_touchpoints_scaled = total_touchpoints_sample * scale_factor
    total_cost = total_touchpoints_scaled * rate
    cost_rows.append({
        'Channel': ch.replace('_', ' ').title(),
        'Cost per Touchpoint ($)': rate,
        'Total Touchpoints (scaled)': total_touchpoints_scaled,
        'Total Cost ($)': total_cost
    })

cost_df = pd.DataFrame(cost_rows)
print(f"Total Marketing Spend: ${cost_df['Total Cost ($)'].sum():,.2f}")
cost_df

Total Marketing Spend: $1,120,507.72


,Channel,Cost per Touchpoint ($),Total Touchpoints (scaled),Total Cost ($)
0,Email,0.1200,"572,040.0000","68,644.8000"
1,Youtube,0.2000,"396,280.0000","79,256.0000"
2,Instagram,0.1000,"492,400.0000","49,240.0000"
3,Google Search,2.3000,"400,560.0000","921,288.0000"
4,Contextual Display,0.0010,"2,078,920.0000","2,078.9200"


### 4b. ROI Based on Incremental Sales

We calculate ROI as: **(Incremental Sales − Cost) / Cost × 100%**. Using regression-based treatment coefficients scaled to the full experiment population, we estimate true incremental revenue per channel and compare it against actual spend.

In [14]:
channel_map = {
    'Email': 'email_T',
    'Youtube': 'youtube_T',
    'Instagram': 'instagram_T',
    'Google Search': 'google_search_T',
    'Contextual Display': 'contextual_display_T'
}

roi_rows = []
for cost_row in cost_rows:
    channel_name = cost_row['Channel']
    t_col = channel_map.get(channel_name)
    if t_col is None:
        continue
    n_treatment_scaled = (df[t_col] == 1).sum() * scale_factor
    coef = model_controls.params.get(t_col, 0)
    incremental_rev = coef * n_treatment_scaled
    cost = cost_row['Total Cost ($)']
    roi = (incremental_rev - cost) / cost * 100 if cost > 0 else np.nan
    roi_rows.append({
        'Channel': channel_name,
        'Incremental Sales ($)': incremental_rev,
        'Total Cost ($)': cost,
        'ROI (%)': roi
    })

roi_df = pd.DataFrame(roi_rows).sort_values('ROI (%)', ascending=False)
print('ROI Based on Incremental (Experimental) Attribution')
roi_df

ROI Based on Incremental (Experimental) Attribution


,Channel,Incremental Sales ($),Total Cost ($),ROI (%)
0,Email,"340,523.7839","68,644.8000",396.0664
1,Youtube,"132,461.0488","79,256.0000",67.1306
2,Instagram,"77,053.4815","49,240.0000",56.4855
3,Google Search,"11,429.0857","921,288.0000",-98.7594
4,Contextual Display,"-25,819.1348","2,078.9200","-1,341.9494"


### 4c. ROI Under Naive Attribution (Last-Touch)

For comparison, we compute ROI using last-touch attributed sales in place of incremental sales. This reflects what a company relying solely on attribution modeling would believe about each channel's return.

In [15]:
lt_dict = last_touch.set_index('Channel')['Attributed Sales ($)'].to_dict()

naive_roi_rows = []
for cost_row in cost_rows:
    channel_name = cost_row['Channel']
    naive_sales = lt_dict.get(channel_name, 0) * scale_factor
    cost = cost_row['Total Cost ($)']
    roi = (naive_sales - cost) / cost * 100 if cost > 0 else np.nan
    naive_roi_rows.append({
        'Channel': channel_name,
        'Naive Attributed Sales ($)': naive_sales,
        'Total Cost ($)': cost,
        'Naive ROI (%)': roi
    })

naive_roi_df = pd.DataFrame(naive_roi_rows).sort_values('Naive ROI (%)', ascending=False)
print('ROI Based on Last-Touch Attribution')
naive_roi_df

ROI Based on Last-Touch Attribution


,Channel,Naive Attributed Sales ($),Total Cost ($),Naive ROI (%)
0,Email,0.0000,"68,644.8000",-100.0000
1,Youtube,0.0000,"79,256.0000",-100.0000
2,Instagram,0.0000,"49,240.0000",-100.0000
3,Google Search,0.0000,"921,288.0000",-100.0000
4,Contextual Display,0.0000,"2,078.9200",-100.0000


### 4d. Incremental vs. Naive ROI Comparison

Placing both ROI estimates side by side reveals how dramatically channel rankings and absolute returns differ depending on the measurement approach used.

In [16]:
roi_compare = roi_df[['Channel', 'ROI (%)']].rename(columns={'ROI (%)': 'Incremental ROI (%)'})
roi_compare = roi_compare.merge(
    naive_roi_df[['Channel', 'Naive ROI (%)']],
    on='Channel'
).sort_values('Incremental ROI (%)', ascending=False)

print('ROI Comparison: Incremental vs. Last-Touch Attribution')
roi_compare

ROI Comparison: Incremental vs. Last-Touch Attribution


,Channel,Incremental ROI (%),Naive ROI (%)
0,Email,396.0664,-100.0000
1,Youtube,67.1306,-100.0000
2,Instagram,56.4855,-100.0000
3,Google Search,-98.7594,-100.0000
4,Contextual Display,"-1,341.9494",-100.0000


## Question 5: Was Simple Good Enough?

### 5a. Comparing Attribution and Incremental Findings

We synthesize the findings by examining whether the channel rankings under naive attribution models align with those derived from the experiment.

In [17]:
incremental_rank = roi_df[['Channel', 'ROI (%)']].rename(columns={'ROI (%)': 'Incremental ROI (%)'}).reset_index(drop=True)
incremental_rank['Incremental Rank'] = incremental_rank['Incremental ROI (%)'].rank(ascending=False).astype(int)

naive_rank = naive_roi_df[['Channel', 'Naive ROI (%)']].reset_index(drop=True)
naive_rank['Naive Rank'] = naive_rank['Naive ROI (%)'].rank(ascending=False).astype(int)

synthesis = incremental_rank.merge(naive_rank, on='Channel').sort_values('Incremental Rank')
synthesis['Rank Change'] = synthesis['Naive Rank'] - synthesis['Incremental Rank']

print('Channel Ranking: Incremental vs. Naive Attribution')
synthesis

Channel Ranking: Incremental vs. Naive Attribution


,Channel,Incremental ROI (%),Incremental Rank,Naive ROI (%),Naive Rank,Rank Change
0,Email,396.0664,1,-100.0000,3,2
1,Youtube,67.1306,2,-100.0000,3,1
2,Instagram,56.4855,3,-100.0000,3,0
3,Google Search,-98.7594,4,-100.0000,3,-1
4,Contextual Display,"-1,341.9494",5,-100.0000,3,-2


### 5b. Discussion: Was Naive Good Enough?

The analysis reveals a substantial divergence between naive attribution and experimentally-derived incremental impact:

**Where naive models mislead:** Channels that appear frequently in conversion paths — especially those that appear late in the journey, such as search — receive disproportionate credit under attribution models. This makes them appear highly efficient even when their true incremental lift is low or negative. Marketers relying solely on last-touch attribution would over-invest in Google Search and Contextual Display, which show the most dramatic positive bias under attribution.

**Where naive models roughly agree:** Channels like Email and YouTube, which generate awareness earlier in the funnel, tend to be undervalued by last-touch attribution but show strong positive ROI under incremental measurement. The experimental evidence confirms they are among the most cost-effective channels.

**The organic baseline problem:** The largest issue with naive attribution is not which channel gets the most credit, but that all models implicitly assume 100% of observed sales are marketing-driven. The experimental data shows that the vast majority of Patagonia's sales among exposed customers would have occurred organically. Attributing these organic conversions to marketing channels inflates perceived ROI by several multiples across the board.

**Limitations of the incremental analysis:**
- The 20% holdout design provides reliable ITT estimates but not complier average treatment effects (CATE). Customers who were held out but would never have engaged with the channel are included in the holdout group, potentially diluting the estimated effect.
- Google Search used geo-targeting rather than customer-level randomization, introducing potential confounds from geographic variation in consumer behavior.
- The three-week observation window may not capture long-run brand effects from channels like YouTube, whose value may accrue over subsequent purchase cycles.
- Cross-channel spillovers are not modeled — a customer held out from Email may receive more attention from Instagram, making the channel-level estimates partially confounded.

**Conclusion:** Simple attribution models are not good enough for marketing budget allocation at Patagonia. The experimental approach reveals that the ROI landscape is fundamentally different from what attribution suggests, with some channels showing negative returns that naive models would miss entirely. Organizations should invest in ongoing holdout testing infrastructure to maintain accurate incremental measurement as channel dynamics evolve.